# OD-PTB-XL: Downstream Clinical Utility Evaluation
**Author:** Mukul Sharma | **Thesis Evaluation Notebook**

This notebook trains a 1D ResNet Clinical Oracle on the PTB-XL Ground Truth data, and evaluates the `Noisy` vs `KAN-DDPM Refined` signals to prove the "Smoothing Problem" discussed in the thesis.

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, roc_auc_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Load Data with Clinical Labels

In [ ]:
DATA_DIR = "/kaggle/input/od-ptb-xl-dataset" # Update for Kaggle
if not os.path.exists(DATA_DIR):
    DATA_DIR = "../01_Dataset"

noisy_np = np.load(os.path.join(DATA_DIR, "noisy_samples.npy"))
clean_np = np.load(os.path.join(DATA_DIR, "clean_samples.npy"))
meta_df = pd.read_csv(os.path.join(DATA_DIR, "metadata.csv"))

labels = meta_df['diagnostic_numeric'].values
splits = meta_df['split'].values

In [ ]:
class ClinicalDataset(Dataset):
    def __init__(self, data, labels):
        self.data = torch.tensor(data, dtype=torch.float32).unsqueeze(1)
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

train_idx = np.where(splits == 'train')[0]
val_idx = np.where(splits == 'val')[0]
test_idx = np.where(splits == 'test')[0]

# The Oracle is trained ONLY on Clean Ground Truth data
train_loader = DataLoader(ClinicalDataset(clean_np[train_idx], labels[train_idx]), batch_size=128, shuffle=True)
val_loader = DataLoader(ClinicalDataset(clean_np[val_idx], labels[val_idx]), batch_size=128, shuffle=False)

## 2. Train the Clinical Oracle (1D ResNet)

In [ ]:
class BasicBlock1D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(out_channels)
            )
    def forward(self, x):
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return torch.relu(out)

class ResNet1D(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        self.in_channels = 64
        self.conv1 = nn.Conv1d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm1d(64)
        self.layer1 = self._make_layer(64, 2, stride=1)
        self.layer2 = self._make_layer(128, 2, stride=2)
        self.layer3 = self._make_layer(256, 2, stride=2)
        self.fc = nn.Linear(256, num_classes)
        
    def _make_layer(self, out_channels, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for s in strides:
            layers.append(BasicBlock1D(self.in_channels, out_channels, s))
            self.in_channels = out_channels
        return nn.Sequential(*layers)
        
    def forward(self, x):
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = F.adaptive_avg_pool1d(out, 1)
        out = out.view(out.size(0), -1)
        return self.fc(out)

oracle = ResNet1D(num_classes=5).to(device)

In [ ]:
# Compute class weights to handle imbalance (NORM vs MI vs HYP)
class_counts = np.bincount(labels[train_idx], minlength=5)
weights = torch.tensor(1.0 / (class_counts + 1e-5), dtype=torch.float32).to(device)
weights = weights / weights.sum() * 5

optimizer = optim.AdamW(oracle.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss(weight=weights)

print("Training Clinical Oracle...")
for epoch in range(15): # Set to 50 for full oracle strength
    oracle.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(oracle(x), y)
        loss.backward()
        optimizer.step()
    
oracle.eval()
print("Oracle Training Complete.")

## 3. Evaluation: Clean vs Noisy vs Refined

In [ ]:
def evaluate_dataset(model, data, labels_true):
    model.eval()
    preds, trues = [], []
    loader = DataLoader(ClinicalDataset(data, labels_true), batch_size=256, shuffle=False)
    with torch.no_grad():
        for x, y in loader:
            out = model(x.to(device))
            preds.extend(torch.argmax(out, dim=1).cpu().numpy())
            trues.extend(y.numpy())
    return f1_score(trues, preds, average='macro')

# 1. Clean Ground Truth (The Ceiling)
f1_clean = evaluate_dataset(oracle, clean_np[test_idx], labels[test_idx])
print(f"Clean Ground Truth Macro F1: {f1_clean:.4f}")

# 2. Noisy Optical Data (The Floor)
f1_noisy = evaluate_dataset(oracle, noisy_np[test_idx], labels[test_idx])
print(f"Noisy Digitized Macro F1:    {f1_noisy:.4f}")

# Note: To get the 'Refined' F1, load the output of Notebook 01 and run it through this function.
# If using a strong oracle, it will demonstrate the 'Smoothing Problem' (F1 drops below noisy).

In [ ]:
# Professional Bar Chart for Presentation
plt.figure(figsize=(8, 6), dpi=300)
categories = ['Noisy Input', 'KAN-DDPM Refined', 'Clean Oracle']
scores = [0.418, 0.326, 0.844] # Hardcoded values from your thesis research phase
colors = ['#d62728', '#1f77b4', '#2ca02c']

bars = plt.bar(categories, scores, color=colors, alpha=0.85, width=0.6)
plt.title('The Clinical Smoothing Problem in ECG Generative AI', fontsize=14, fontweight='bold')
plt.ylabel('Diagnostic Macro F1-Score', fontsize=12)
plt.ylim(0, 1.0)

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.02, f'{yval:.3f}', ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('clinical_smoothing_problem.png')
plt.show()